# **Olist ecommerce exploratory analysis**
---
https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

This notebook reads the cleaned tables the previous notebook wrote to `data_clean/` and asks them a small set of questions, chosen for the ML service that will be built on this data in the near future. Which problem that service solves is not settled before this notebook. It is settled here, from what the questions below find.

**Delivery** is the candidate the questions test. It is the problem where a prediction is needed at checkout, where everything the service would use is already known at that moment, and where the data carries a benchmark of its own, the delivery date Olist estimated for every order. The first question checks the alternatives before that candidate is examined.

This is a light pass. Each question gets one or two plots or counts and a verdict of two lines, and a result is followed further only when it comes out large and unexplained. Where a question needs a rule, such as which orders form the population or which distance an order with more than one seller takes, the rule is declared before the count, as in the cleaning notebook. Every rule the cleaning notebook's summary carries into the analysis notebook applies here.

### How this notebook is organised

| Section | Question | What it decides for the service |
|---|---|---|
| Loading the cleaned tables | Do the tables arrive as the cleaning notebook wrote them? | - |
| Q0. Which problem the data supports | How many customers order more than once, and how many reviews are written before their order arrives? | Whether a problem per customer, such as churn, lifetime value or recommendation, or the review score could replace delivery as the target |
| Q1. Delivery time | What shape does the time from purchase to delivery take? | The error metric and whether the target needs a transform. The upper tail is kept, since a promised date is read from it |
| Q2. Distance | Does the distance between customer and seller explain delivery time? | Whether distance earns its place as a feature, and with it the purpose of the geolocation aggregation in the cleaning notebook |
| Q3. Olist's estimate | How far is the delivery Olist estimated from the delivery that happened, and how often is it late? | The baseline, and whether the service predicts an expected time or a promise that is kept as often as Olist's and is shorter than it |
| Q4. Delivery over time | How do order volume and delivery time move from month to month? | Whether the split follows time or is random, and which months at the edges stay out of training |
| Summary | What the questions found | The target, the population, the baseline, the split and the features carried into the service |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

### **Loading the cleaned tables**

The nine tables are read from `data_clean/` as the cleaning notebook wrote them, with no conversion. The cleaning notebook already read every file back and found it identical, so what is checked here is only that nothing has changed since. Each table's shape is compared with the shape the export produced, and each group of columns with the dtype it was written with, timestamps as datetime, identifiers as text, zip code prefixes as integers and the four order flags as booleans. The Distance section builds its distances by merging on `zip_code_prefix`, so the check also confirms that `geolocation_by_prefix` still holds one row per prefix.

In [2]:
data_clean_dir = Path("data_clean")

table_names = ["customers", "order_items", "order_payments", "order_reviews", "orders",
               "products", "sellers", "prod_categ_name_transl", "geolocation_by_prefix"]

In [3]:
dataframes = {}
for name in table_names:
    dataframes[name] = pd.read_parquet(data_clean_dir / f"{name}.parquet")

for name, df in dataframes.items():
    print(f"{name}: shape = {df.shape}")

customers: shape = (99441, 5)
order_items: shape = (112650, 7)
order_payments: shape = (103886, 5)
order_reviews: shape = (99224, 7)
orders: shape = (99441, 12)
products: shape = (32951, 9)
sellers: shape = (3095, 4)
prod_categ_name_transl: shape = (73, 2)
geolocation_by_prefix: shape = (18901, 6)


Each table's shape is printed in the same form the export printed it, so the two can be read side by side. The same comparison is then made in code, so that a later run shows a mismatch without anyone having to read the two lists.

In [4]:
expected_shapes = {
    "customers": (99441, 5),
    "order_items": (112650, 7),
    "order_payments": (103886, 5),
    "order_reviews": (99224, 7),
    "orders": (99441, 12),
    "products": (32951, 9),
    "sellers": (3095, 4),
    "prod_categ_name_transl": (73, 2),
    "geolocation_by_prefix": (18901, 6),
}

In [5]:
for name, expected in expected_shapes.items():
    got = dataframes[name].shape
    print(f"{name}: expected = {expected}, got = {got}, match = {got == expected}")

customers: expected = (99441, 5), got = (99441, 5), match = True
order_items: expected = (112650, 7), got = (112650, 7), match = True
order_payments: expected = (103886, 5), got = (103886, 5), match = True
order_reviews: expected = (99224, 7), got = (99224, 7), match = True
orders: expected = (99441, 12), got = (99441, 12), match = True
products: expected = (32951, 9), got = (32951, 9), match = True
sellers: expected = (3095, 4), got = (3095, 4), match = True
prod_categ_name_transl: expected = (73, 2), got = (73, 2), match = True
geolocation_by_prefix: expected = (18901, 6), got = (18901, 6), match = True


Each group of columns is checked against the single dtype the export wrote. The timestamps are expected at microsecond resolution, `datetime64[us]`, which is the resolution pandas chose when the cleaning notebook parsed them, so a change of resolution counts as a change.

In [6]:
dtype_groups = {
    "timestamps": ("datetime64[us]", [
        ("orders", "order_purchase_timestamp"),
        ("orders", "order_approved_at"),
        ("orders", "order_delivered_carrier_date"),
        ("orders", "order_delivered_customer_date"),
        ("orders", "order_estimated_delivery_date"),
        ("order_items", "shipping_limit_date"),
        ("order_reviews", "review_creation_date"),
        ("order_reviews", "review_answer_timestamp"),
    ]),

    "identifiers": ("str", [
        ("customers", "customer_id"),
        ("customers", "customer_unique_id"),
        ("orders", "order_id"),
        ("orders", "customer_id"),
        ("order_items", "order_id"),
        ("order_items", "product_id"),
        ("order_items", "seller_id"),
        ("order_payments", "order_id"),
        ("order_reviews", "review_id"),
        ("order_reviews", "order_id"),
        ("products", "product_id"),
        ("sellers", "seller_id"),
    ]),

    "zip code prefixes": ("int64", [
        ("customers", "customer_zip_code_prefix"),
        ("sellers", "seller_zip_code_prefix"),
        ("geolocation_by_prefix", "zip_code_prefix"),
    ]),

    "order flags": ("bool", [
        ("orders", "carrier_before_purchase"),
        ("orders", "carrier_before_approval"),
        ("orders", "delivered_before_approval"),
        ("orders", "delivered_before_carrier"),
    ]),
}

In [7]:
for group, (expected, columns) in dtype_groups.items():
    got = set()
    for table, column in columns:
        got.add(str(dataframes[table][column].dtype))
    print(f"{group}: expected = {{'{expected}'}}, got = {got}, match = {got == {expected}}")

timestamps: expected = {'datetime64[us]'}, got = {'datetime64[us]'}, match = True
identifiers: expected = {'str'}, got = {'str'}, match = True
zip code prefixes: expected = {'int64'}, got = {'int64'}, match = True
order flags: expected = {'bool'}, got = {'bool'}, match = True


In [8]:
got = dataframes["geolocation_by_prefix"]["zip_code_prefix"].is_unique
print(f"zip_code_prefix unique: expected = True, got = {got}, match = {got == True}")

zip_code_prefix unique: expected = True, got = True, match = True


All nine tables arrive with the shapes the export wrote, each of the four groups of columns holds the dtype it was written with, and `geolocation_by_prefix` holds one row per prefix. None of these columns needs converting, and the Distance section can merge on `zip_code_prefix` without deduplicating first.